# Axe 3 — RFormer on Colab T4

Runs `experiments/chap5/5_4_1_rformer.py` on Colab's GPU runtime
from PyCharm's Jupyter integration.

**Prereq**: connect this notebook's Jupyter server to Google Colab in
PyCharm (Settings → Languages & Frameworks → Jupyter → Add Colab), then
choose the T4 runtime.

**Filesystem note**: cells execute on Colab's VM, not your laptop.
The repo is cloned from GitHub in the first cell so all local paths
(`data/processed/`, `results/chap5/`) resolve against the clone.

## 1. Environment check

Verify the GPU is attached and install the two deps not preinstalled
on Colab (`iisignature`; `torch` and the rest are already there).

In [1]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

!pip install -q iisignature

torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 63.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 2. Fetch the repo

Clones on first run, pulls latest on subsequent runs. All later cells
assume the working directory is the repo root.

In [ ]:
import os
from google.colab import userdata

GH_USER = 'cnammastermlballa-dotcom'
GH_REPO = 'path-signature-benchmark'

# Try Colab's secrets store first. NOTE: userdata does NOT work from
# PyCharm-Colab — it only works in the Colab web UI. In PyCharm, use
# the getpass fallback below.
token = None
try:
    token = userdata.get('GH_TOKEN')
    print("Loaded GH_TOKEN from Colab secrets.")
except userdata.SecretNotFoundError:
    print("SecretNotFoundError: no secret named 'GH_TOKEN' exists.")
except userdata.NotebookAccessError:
    print("NotebookAccessError: secret exists but not granted to this notebook.")
except Exception as e:
    print(f"Cannot fetch Colab secret ({type(e).__name__}) — "
          "expected when running from PyCharm-Colab.")

# Interactive fallback: paste the PAT once, keep it in memory only.
if not token:
    import getpass
    typed = getpass.getpass(
        "Paste GH_TOKEN (hidden input), or press Enter to try public clone: "
    ).strip()
    if typed:
        token = typed

if token:
    REPO_URL = f'https://{token}@github.com/{GH_USER}/{GH_REPO}.git'
    print("Using token-authenticated clone URL.")
else:
    REPO_URL = f'https://github.com/{GH_USER}/{GH_REPO}.git'
    print("No token — falling back to public URL.")

if not os.path.exists(GH_REPO):
    !git clone {REPO_URL}
%cd {GH_REPO}
!git pull --ff-only

## 3. Run the Axe 3 pipeline

The script reads `optimal_N_{ds}.txt` and `best_config_{ds}.txt` from
`results/chap5/`, prepares multi-view signatures for CharTraj + NATOPS,
trains RFormer, evaluates on the held-out test set, and saves
`results/chap5/5_4_1_rformer.{csv,png}`.

Expected wall time on T4: ~15-30 min (CharTraj dominates).

In [4]:
!python experiments/chap5/5_4_1_rformer.py

python3: can't open file '/content/path-signature-benchmark/experiments/chap5/5_4_1_rformer.py': [Errno 2] No such file or directory


## 4. Inspect the results

In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv('results/chap5/5_4_1_rformer.csv')
display(df)
display(Image('results/chap5/5_4_1_rformer.png'))

## 5. Commit results back to GitHub (optional)

Set your git identity and use a Personal Access Token so the push
authenticates non-interactively. **Do not paste the token into a cell
you commit** — set it as a Colab secret.

In Colab: `Tools → Manage secrets` and add `GH_TOKEN` (a fine-grained
PAT with `contents: write` on this repo).

In [ ]:
from google.colab import userdata
import subprocess

GH_USER = 'cnammastermlballa-dotcom'
GH_REPO = 'path-signature-benchmark'
token = userdata.get('GH_TOKEN')

!git config user.name  'Thierry Balla (Colab)'
!git config user.email 'cnammastermlballa@gmail.com'
subprocess.run(
    ['git', 'remote', 'set-url', 'origin',
     f'https://{token}@github.com/{GH_USER}/{GH_REPO}.git'],
    check=True,
)

!git add results/chap5/5_4_1_rformer.csv results/chap5/5_4_1_rformer.png
!git commit -m 'Axe 3 results (Colab T4 run)' && git push